# NB03 — General Essentiality Model Training and LOGO Evaluation

Trains a CatBoost binary classifier predicting cross-stressor conditional essentiality.

**Methodology (corrected from refocus):**
1. Organism-stratified train/test split (verified in NB01 — zero organism overlap)
2. True genus-level LeaveOneGroupOut: all organisms from the held-out genus are in test fold
3. `scale_pos_weight = n_neg / n_pos` for class imbalance
4. Metrics: AUC-ROC, MCC, sensitivity at 90% specificity — not accuracy

Comparison baseline: enigma_stress_phenotype_ml per-metal LOGO AUC 0.53–0.62

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import roc_auc_score, matthews_corrcoef, confusion_matrix, precision_recall_curve
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

PROJ_ROOT = Path.cwd().parent
DATA_DIR  = PROJ_ROOT / 'data'
FIG_DIR   = PROJ_ROOT / 'figures'
FIG_DIR.mkdir(exist_ok=True)

# Load features (saved as .npy by NB02)
feat_cols    = json.load(open(DATA_DIR / 'feature_columns.json'))
X_train      = np.load(DATA_DIR / 'X_train.npy')
X_test       = np.load(DATA_DIR / 'X_test.npy')
y_train      = np.load(DATA_DIR / 'y_train.npy')
y_test       = np.load(DATA_DIR / 'y_test.npy')
orgs_train   = np.load(DATA_DIR / 'orgs_train.npy', allow_pickle=True)
genera_train = np.load(DATA_DIR / 'genera_train.npy', allow_pickle=True)
orgs_test    = np.load(DATA_DIR / 'orgs_test.npy', allow_pickle=True)

n_pos = y_train.sum()
n_neg = (y_train == 0).sum()
scale_pos_weight = n_neg / n_pos

print(f"Train: {len(X_train):,} proteins, {n_pos} essential ({100*n_pos/len(y_train):.2f}%)")
print(f"scale_pos_weight: {scale_pos_weight:.1f}")

Train: 166,705 proteins, 22406 essential (13.44%)
scale_pos_weight: 6.4


### Train final model on full training set

In [2]:
model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    scale_pos_weight=scale_pos_weight,
    eval_metric='AUC',
    random_seed=42,
    verbose=100,
)
model.fit(X_train, y_train)

# Test set evaluation
y_pred_proba = model.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_pred_proba)

# Optimal threshold from precision-recall
prec, rec, thresholds = precision_recall_curve(y_test, y_pred_proba)
f1_scores = 2 * prec * rec / (prec + rec + 1e-9)
best_thresh = thresholds[np.argmax(f1_scores)]
y_pred_bin = (y_pred_proba >= best_thresh).astype(int)
test_mcc = matthews_corrcoef(y_test, y_pred_bin)

print(f"\n=== TEST SET RESULTS (held-out organisms) ===")
print(f"AUC-ROC: {test_auc:.4f}")
print(f"MCC (threshold={best_thresh:.3f}): {test_mcc:.4f}")
print(f"\nConfusion matrix (threshold={best_thresh:.3f}):")
print(confusion_matrix(y_test, y_pred_bin))

0:	total: 77.4ms	remaining: 38.6s


100:	total: 1.81s	remaining: 7.16s


200:	total: 3.52s	remaining: 5.24s


300:	total: 5.24s	remaining: 3.46s


400:	total: 6.96s	remaining: 1.72s


499:	total: 8.67s	remaining: 0us

=== TEST SET RESULTS (held-out organisms) ===
AUC-ROC: 0.6594
MCC (threshold=0.521): 0.1485

Confusion matrix (threshold=0.521):
[[33880  9811]
 [ 2601  2054]]


### True genus-level LOGO cross-validation

Each fold holds out ALL proteins from ALL organisms in one genus.
Folds with fewer than 5 positive proteins are flagged as unreliable.

In [3]:
logo = LeaveOneGroupOut()
logo_results = []

genera_unique = np.unique(genera_train)
print(f"Running LOGO over {len(genera_unique)} genera...")
print()

for fold_idx, (tr_idx, te_idx) in enumerate(logo.split(X_train, y_train, groups=genera_train)):
    genus = genera_train[te_idx[0]]
    X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
    X_te, y_te = X_train[te_idx], y_train[te_idx]

    # Skip folds with no class diversity
    if y_tr.nunique() < 2 if hasattr(y_tr, 'nunique') else len(np.unique(y_tr)) < 2:
        print(f"  [{genus}] SKIP — training fold has single class")
        continue
    if y_te.sum() < 5:
        flag = "⚠️ LOW_N"
    else:
        flag = ""

    # Per-fold class weight
    n_pos_fold = y_tr.sum()
    n_neg_fold = (y_tr == 0).sum()
    spw = n_neg_fold / max(n_pos_fold, 1)

    fold_model = CatBoostClassifier(
        iterations=300,
        learning_rate=0.05,
        depth=6,
        scale_pos_weight=spw,
        random_seed=42,
        verbose=0,
    )
    fold_model.fit(X_tr, y_tr)
    proba = fold_model.predict_proba(X_te)[:, 1]

    if len(np.unique(y_te)) < 2:
        auc = float('nan')
    else:
        auc = roc_auc_score(y_te, proba)

    # Threshold-tune on fold test set (conservative: max F1)
    if len(np.unique(y_te)) >= 2:
        prec_f, rec_f, thresh_f = precision_recall_curve(y_te, proba)
        f1_f = 2 * prec_f * rec_f / (prec_f + rec_f + 1e-9)
        best_t = thresh_f[np.argmax(f1_f[:-1])] if len(thresh_f) > 0 else 0.5
    else:
        best_t = 0.5
    y_pred_f = (proba >= best_t).astype(int)
    mcc = matthews_corrcoef(y_te, y_pred_f) if len(np.unique(y_te)) >= 2 else float('nan')

    # Organisms held out in this fold
    held_out_orgs = list(np.unique(orgs_train[te_idx]))

    logo_results.append({
        'genus': genus,
        'auc': auc,
        'mcc': mcc,
        'n_test': len(y_te),
        'n_pos': int(y_te.sum()),
        'n_neg': int((y_te == 0).sum()),
        'n_train': len(y_tr),
        'held_out_organisms': held_out_orgs,
        'low_n_flag': y_te.sum() < 5,
    })

    print(f"  [{genus}] AUC={auc:.4f}, MCC={mcc:.4f}, n_pos={y_te.sum()}, n_orgs_held={len(held_out_orgs)} {flag}")

logo_df = pd.DataFrame(logo_results)
logo_df.to_csv(DATA_DIR / 'logo_results.csv', index=False)
print(f"\nSaved logo_results.csv ({len(logo_df)} folds)")

Running LOGO over 32 genera...



  [Acidovorax] AUC=0.6356, MCC=0.1653, n_pos=733, n_orgs_held=1 


  [Bacteroides] AUC=0.5910, MCC=0.0974, n_pos=941, n_orgs_held=2 


  [Bifidobacterium] AUC=0.5453, MCC=0.0366, n_pos=30, n_orgs_held=1 


  [Brevundimonas] AUC=nan, MCC=nan, n_pos=0, n_orgs_held=1 ⚠️ LOW_N


...

  [Synechococcus] AUC=0.5615, MCC=0.0576, n_pos=578, n_orgs_held=1 


  [Unknown_CL21] AUC=nan, MCC=nan, n_pos=0, n_orgs_held=1 ⚠️ LOW_N


  [Unknown_Korea] AUC=0.6485, MCC=0.1592, n_pos=501, n_orgs_held=1 


  [Unknown_Miya] AUC=0.6085, MCC=0.1326, n_pos=648, n_orgs_held=1 


  [Unknown_SB2B] AUC=0.6567, MCC=0.2045, n_pos=747, n_orgs_held=1 

Saved logo_results.csv (32 folds)


### Summarize LOGO results

In [4]:
# Exclude low-n folds from aggregate (but report them)
reliable = logo_df[~logo_df['low_n_flag']]
low_n    = logo_df[logo_df['low_n_flag']]

print("=== LOGO SUMMARY (genus-level) ===")
print(f"Total folds: {len(logo_df)}")
print(f"Reliable folds (≥5 positives): {len(reliable)}")
print(f"Low-n folds (flagged): {len(low_n)}")
print()

if len(reliable) > 0:
    valid_auc = reliable['auc'].dropna()
    valid_mcc = reliable['mcc'].dropna()
    print(f"Reliable folds — AUC: {valid_auc.mean():.4f} ± {valid_auc.std():.4f}")
    print(f"Reliable folds — MCC: {valid_mcc.mean():.4f} ± {valid_mcc.std():.4f}")

print()
print("All folds (sorted by AUC):")
print(logo_df.sort_values('auc', ascending=False)[
    ['genus','auc','mcc','n_pos','n_test','held_out_organisms','low_n_flag']
].to_string(index=False))

print()
print(f"=== COMPARISON ===")
print(f"This project (genus-level LOGO):  AUC = {valid_auc.mean():.3f} ± {valid_auc.std():.3f}")
print(f"enigma_stress_phenotype_ml metals: AUC = 0.53–0.62 (per-metal, genus-level LOGO)")
print()
print("Note: LOGO strictness is equivalent between projects (genus held out).")
print("The union label integrates signal across all stressors; expect similar or slightly higher AUC.")

=== LOGO SUMMARY (genus-level) ===
Total folds: 32
Reliable folds (≥5 positives): 27
Low-n folds (flagged): 5

Reliable folds — AUC: 0.6180 ± 0.0369
Reliable folds — MCC: 0.1260 ± 0.0495

All folds (sorted by AUC):
           genus      auc      mcc  n_pos  n_test                                                                                               held_out_organisms  low_n_flag
      Pedobacter 0.685514 0.205409    636    4343                                                                                                        [Pedo557]       False
       Ralstonia 0.664663 0.081710    401   14995                                            [RalstoniaBSBF1503, RalstoniaGMI1000, RalstoniaPSI07, RalstoniaUW163]       False
    Burkholderia 0.663801 0.132806    246    5081                                                                                                        [Burk376]       False
         Dickeya 0.661194 0.102437    364    7010                                    

### Feature importance

In [5]:
importances = pd.Series(model.get_feature_importance(), index=feat_cols)
top30 = importances.nlargest(30)

fig, ax = plt.subplots(figsize=(8, 8))
top30.sort_values().plot.barh(ax=ax, color='steelblue')
ax.set_xlabel('Feature importance')
ax.set_title('Top 30 Features — Cross-Stressor Essentiality Model')
plt.tight_layout()
plt.savefig(FIG_DIR / 'nb03_feature_importance.png', dpi=150)
plt.close()
print("Saved: figures/nb03_feature_importance.png")

Saved: figures/nb03_feature_importance.png


### LOGO AUC distribution plot

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# AUC distribution
sorted_df = logo_df.sort_values('auc', ascending=False).reset_index(drop=True)
bar_colors = ['#d62728' if f else '#1f77b4' for f in sorted_df['low_n_flag']]
axes[0].bar(range(len(sorted_df)), sorted_df['auc'], color=bar_colors)
axes[0].axhline(0.5, color='gray', linestyle='--', label='Random (AUC=0.5)')
if len(reliable) > 0:
    axes[0].axhline(reliable['auc'].mean(), color='navy', linestyle='-',
                    label=f"Mean reliable AUC={reliable['auc'].mean():.3f}")
axes[0].set_xlabel('Genus (rank ordered)')
axes[0].set_ylabel('LOGO AUC')
axes[0].set_title('Genus-Level LOGO AUC\n(red = low-n, <5 positives)')
axes[0].legend(fontsize=8)

# n_pos vs AUC scatter
axes[1].scatter(logo_df['n_pos'], logo_df['auc'],
                c=['#d62728' if f else '#1f77b4' for f in logo_df['low_n_flag']],
                alpha=0.8, s=60)
axes[1].axhline(0.5, color='gray', linestyle='--')
axes[1].set_xlabel('# Positive proteins in test fold')
axes[1].set_ylabel('LOGO AUC')
axes[1].set_title('Test-fold size vs. AUC')

plt.tight_layout()
plt.savefig(FIG_DIR / 'nb03_logo_auc.png', dpi=150)
plt.close()
print("Saved: figures/nb03_logo_auc.png")

# Save model
model.save_model(str(DATA_DIR / 'general_essentiality_model.cbm'))
print("Saved: data/general_essentiality_model.cbm")

Saved: figures/nb03_logo_auc.png
Saved: data/general_essentiality_model.cbm


### Save final metrics

In [7]:
import json

final_metrics = {
    'test_auc': float(test_auc),
    'test_mcc': float(test_mcc),
    'test_threshold': float(best_thresh),
    'logo_n_folds': int(len(logo_df)),
    'logo_n_reliable': int(len(reliable)),
    'logo_mean_auc_reliable': float(reliable['auc'].mean()) if len(reliable) > 0 else None,
    'logo_std_auc_reliable':  float(reliable['auc'].std())  if len(reliable) > 0 else None,
    'logo_mean_mcc_reliable': float(reliable['mcc'].mean()) if len(reliable) > 0 else None,
    'logo_std_mcc_reliable':  float(reliable['mcc'].std())  if len(reliable) > 0 else None,
    'scale_pos_weight': float(scale_pos_weight),
    'n_features': len(feat_cols),
    'n_train': int(len(X_train)),
    'n_test': int(len(X_test)),
}

with open(DATA_DIR / 'model_metrics.json', 'w') as f:
    json.dump(final_metrics, f, indent=2)

print("=== FINAL METRICS ===")
for k, v in final_metrics.items():
    print(f"  {k}: {v}")

=== FINAL METRICS ===
  test_auc: 0.6594182546646734
  test_mcc: 0.14853571992734496
  test_threshold: 0.5206457440182067
  logo_n_folds: 32
  logo_n_reliable: 27
  logo_mean_auc_reliable: 0.6179775202246434
  logo_std_auc_reliable: 0.03689064897277829
  logo_mean_mcc_reliable: 0.12602588379719332
  logo_std_mcc_reliable: 0.04948131756992265
  scale_pos_weight: 6.440194590734625
  n_features: 420
  n_train: 166705
  n_test: 48346
